In [1]:
import os
os.getcwd()
os.chdir("..")
os.getcwd()

'/home/thienhb/Workspace/arxiv-paper-rag'

In [5]:
from src.services.indexing.text_chunker import TextChunker

In [6]:
sample_text = """
1. Introduction to Natural Language Processing
Natural Language Processing (NLP) is a subfield of artificial intelligence, computer science, and linguistics concerned with the interactions between computers and human language. The ultimate goal of NLP is to enable computers to understand, interpret, and generate human languages in a way that is valuable.

2. Understanding Text Chunking Strategies
Chunking is the process of splitting large documents into smaller, manageable segments prior to embedding and indexing in a vector database. Choosing the right chunk size and overlap is crucial for Retrieval-Augmented Generation (RAG) systems. If chunks are too small, they lose critical context. If chunks are too large, they dilute the relevance of search results.

3. The Role of Overlap in Context Retention
Adding an overlap between consecutive chunks ensures that semantic meaning is not lost at the boundaries. Without overlap, a key sentence split across two chunks might lose its context, causing retrieval mechanisms to miss relevant information. A standard rule of thumb is an overlap of 10% to 20% of the target chunk size.

4. Evaluating Chunk Quality
When testing your chunking configuration, evaluate whether individual chunks remain semantically self-contained. High-quality chunks retain clear subject references and allow embeddings to accurately capture the intent of the underlying text for similarity search.
"""

In [8]:
text_chunker = TextChunker(chunk_size=600, overlap_size=100)

In [14]:
words = text_chunker._split_into_words(sample_text)
words[:10]  # Display the first 10 words to verify splitting

['1.',
 'Introduction',
 'to',
 'Natural',
 'Language',
 'Processing',
 'Natural',
 'Language',
 'Processing',
 '(NLP)']

In [16]:
sentence = text_chunker._reconstruct_text(words)
sentence[:100]  # Display the first 100 characters of the reconstructed text to verify correctness

'1. Introduction to Natural Language Processing Natural Language Processing (NLP) is a subfield of ar'

### Example 1: Basic Word-Based Chunking (chunk_text)

This example uses traditional word-based chunking with custom chunk_size and overlap_size.

In [17]:
# Instantiate chunker: 50-word chunks with 10-word overlap
chunker = TextChunker(chunk_size=50, overlap_size=10, min_chunk_size=20)

sample_paper_text = """
Quantum computing is a rapidly emerging technology that harnesses the laws of quantum mechanics to solve problems 
too complex for classical computers. Today, IBM Quantum makes real quantum hardware available to thousands of developers. 
Our scientists lead the advancement of quantum computing by designing devices, developing software, and integrating 
classical and quantum computational frameworks.

Qubits are the fundamental unit of information in quantum computing, analogous to the bit in classical computing. 
However, while a classical bit can only be 0 or 1, a qubit can exist in a superposition of states. This capability, 
combined with quantum entanglement, allows quantum systems to process vast amounts of possibilities simultaneously. 
As quantum algorithms evolve, they promise breakthroughs in cryptography, materials science, and complex optimization problems.
"""

chunks = chunker.chunk_text(
    text=sample_paper_text,
    arxiv_id="2401.00001",
    paper_id="paper_123"
)

print(f"Total Chunks Created: {len(chunks)}\n")
for chunk in chunks:
    meta = chunk.metadata
    print(f"--- Chunk #{meta.chunk_index} | Words: {meta.word_count} | Title: {meta.section_title} ---")
    print(f"Overlaps -> Prev: {meta.overlap_with_previous} words | Next: {meta.overlap_with_next} words")
    print(f"Content Preview: {chunk.text[:120]}...\n")

Total Chunks Created: 3

--- Chunk #0 | Words: 50 | Title: None ---
Overlaps -> Prev: 0 words | Next: 10 words
Content Preview: Quantum computing is a rapidly emerging technology that harnesses the laws of quantum mechanics to solve problems too co...

--- Chunk #1 | Words: 50 | Title: None ---
Overlaps -> Prev: 10 words | Next: 10 words
Content Preview: quantum computing by designing devices, developing software, and integrating classical and quantum computational framewo...

--- Chunk #2 | Words: 42 | Title: None ---
Overlaps -> Prev: 10 words | Next: 0 words
Content Preview: or 1, a qubit can exist in a superposition of states. This capability, combined with quantum entanglement, allows quantu...



### Example 2: Section-Based Hybrid Chunking (chunk_paper)

This tests the chunk_paper method where sections of varying sizes (small, optimal 100–800 words, and large >800 words) are processed dynamically.

In [19]:
chunker = TextChunker(chunk_size=100, overlap_size=20, min_chunk_size=30)

title = "Advancements in Neural Network Pruning Techniques"
abstract = "Neural network pruning reduces model size while retaining accuracy. We survey current state-of-the-art method."

sections = {
    # Small section (<100 words) -> Will be merged with adjacent small sections
    "1. Introduction": "Deep learning models have grown exponentially in parameter size over recent years.",
    
    # Small section (<100 words) -> Will combine with section 1
    "2. Motivation": "Deploying large models on edge devices requires significant memory optimization.",
    
    # Medium section (100–800 words) -> Kept as a single chunk with Title + Abstract header attached
    "3. Pruning Methodologies": (
        "Pruning methods can be broadly classified into structured and unstructured pruning. "
        "Unstructured pruning zeroes out individual weights based on magnitude thresholds, leading to sparse matrices. "
        "Structured pruning removes entire channels, neurons, or layers, directly reducing runtime latency without "
        "requiring specialized sparse-matrix hardware acceleration. "
    ) * 10,  # ~150 words
}

chunks = chunker.chunk_paper(
    title=title,
    abstract=abstract,
    full_text="",  # Ignored when sections are valid
    arxiv_id="2402.99999",
    paper_id="paper_456",
    sections=sections
)

print(f"Section Chunks Created: {len(chunks)}\n")
for c in chunks:
    print(f"== Chunk {c.metadata.chunk_index} [{c.metadata.section_title}] ==")
    print(f"Word Count: {c.metadata.word_count}")
    print(f"Text:\n{c.text[:250]}...\n")

Section Chunks Created: 2

== Chunk 0 [1. Introduction + 2. Motivation] ==
Word Count: 48
Text:
Advancements in Neural Network Pruning Techniques

Abstract: Neural network pruning reduces model size while retaining accuracy. We survey current state-of-the-art method.

Section: 1. Introduction

Deep learning models have grown exponentially in pa...

== Chunk 1 [3. Pruning Methodologies] ==
Word Count: 455
Text:
Advancements in Neural Network Pruning Techniques

Abstract: Neural network pruning reduces model size while retaining accuracy. We survey current state-of-the-art method.

Section: 3. Pruning Methodologies

Pruning methods can be broadly classified ...



### Example 3: Section Chunking with JSON String Input

Your code supports JSON strings for sections. Here is how to pass structured section lists as raw JSON.

In [20]:
import json

chunker = TextChunker(chunk_size=150, overlap_size=30)

json_sections = json.dumps([
    {
        "title": "Abstract Duplicate Test",
        "content": "Neural network pruning reduces model size while retaining accuracy."  # Duplicate abstract (will be skipped by _filter_sections)
    },
    {
        "title": "Methodology",
        "content": "We propose a weight-decay dynamic thresholding algorithm that adapts during backpropagation." * 8
    },
    {
        "title": "Conclusion",
        "content": "Dynamic thresholding improves sparsity by 15% without sacrificing accuracy."
    }
])

chunks = chunker.chunk_paper(
    title="Dynamic Weight Decay in Deep Networks",
    abstract="Neural network pruning reduces model size while retaining accuracy.",
    full_text="Full text fallback...",
    arxiv_id="2403.11111",
    paper_id="paper_789",
    sections=json_sections
)

print(f"Resulting Chunks: {len(chunks)}")
for chunk in chunks:
    print(f"- Section: {chunk.metadata.section_title} ({chunk.metadata.word_count} words)")

Resulting Chunks: 1
- Section: Methodology + Conclusion (109 words)
